# AIAT 125 — Unit 1: AI Model Deployment Lifecycle
## Guided Lab: Model Versioning, Checkpointing, Latency Testing & Safety Filters

**Course**: AIAT 125 | **Institution**: Tuwaiq Academy, AI Diploma  
**Unit**: 1 — AI Model Deployment Lifecycle  
**Total Points**: 100

---

### What You Will Do

In this lab you will practise four core skills that every ML engineer needs before pushing a model to production:

| # | Topic | Why It Matters |
|---|-------|----------------|
| 1 | **Model Versioning & Checkpoints** | Enables safe rollback when a new version degrades |
| 2 | **Automated Checkpointing** | Prevents losing the best weights during a long training run |
| 3 | **Model Registry** | Acts as a deployment gate — only staged models can go to production |
| 4 | **Latency Benchmarking** | A model that is accurate but too slow breaks the user experience |
| 5 | **Safety Content Filters** | Stops harmful outputs before they reach end-users |

### How to Use This Notebook

- **Setup cells** (grey header): run them as-is — they install libraries and define helpers.
- **Task cells** (blue header): look for `# TODO` comments and replace `pass` with your code.
- **Assertion cells** (green header): run them after each task — they will tell you immediately if something is wrong.
- **Final Gate**: the last cell prints PASS / FAIL for every task.

> **Tip**: If you get confused, re-read the concept section above each task before touching the code.

---

## Setup — Run This First

This cell imports every library you will need. All of them come pre-installed in standard Python/Conda environments with PyTorch.

In [ ]:
import torch
import torch.nn as nn
import json
import time
import os
import numpy as np

# Use /tmp/ for all file I/O to avoid permission issues on any OS
SAVE_DIR = "/tmp/aiat125_unit1/"
os.makedirs(SAVE_DIR, exist_ok=True)

print("Setup complete.")
print(f"PyTorch version : {torch.__version__}")
print(f"Save directory  : {SAVE_DIR}")

---
## Concept 1 — Model Versioning with Checkpoints

### Why does versioning matter?

When you deploy a model, bugs are inevitable. A new model version may improve accuracy on your test set but perform worse on live traffic (a phenomenon called **distribution shift**). Without version control you have no way to roll back quickly.

The industry solution is to **save model weights together with structured metadata** — version tag, training loss, framework version, and author. This bundle is called a **checkpoint**.

### Pattern from the Unit 1 PDF

In [ ]:
# ── SETUP: Concept Demo ──────────────────────────────────────────────────────
# Run this cell to see how versioned checkpoints are built.
# You do NOT need to modify it.

demo_model = nn.Linear(10, 2)

checkpoint_weights = demo_model.state_dict()

metadata = {
    "version": "v1.4.2",
    "developer": "AI_Team_Alpha",
    "training_loss": 0.045,
    "framework": "PyTorch 2.0"
}

demo_save_path = os.path.join(SAVE_DIR, "model_v1_4_2.pth")

torch.save({
    'model_state': checkpoint_weights,
    'metadata': metadata
}, demo_save_path)

print(f"Demo checkpoint saved → {demo_save_path}")
print(f"Keys inside the file : {list(torch.load(demo_save_path, weights_only=False).keys())}")
print(f"Metadata             : {metadata}")

---
## Concept 2 — Automated Checkpointing (Save Only on Improvement)

### Why not save every epoch?

Training a large model can take days. Saving after every epoch wastes disk space and I/O time. The standard practice is **early-stopping checkpointing**: only overwrite the saved file when the current epoch's loss is strictly better than the best seen so far.

This guarantees that when training finishes (or crashes), you always have the best-performing snapshot on disk.

### Pattern from the Unit 1 PDF

In [ ]:
# ── SETUP: Concept Demo ──────────────────────────────────────────────────────

def auto_checkpoint(epoch, model_state, current_loss, best_loss=0.50):
    """Save checkpoint only when current_loss improves on best_loss."""
    if current_loss < best_loss:
        filename = os.path.join(SAVE_DIR, f"checkpoint_epoch_{epoch}.pth")
        torch.save({'model_state': model_state, 'epoch': epoch, 'loss': current_loss}, filename)
        print(f"  New best model found at epoch {epoch}! Saving {filename}...")
        return True
    return False

# Quick demo: simulate 5 epochs with decreasing loss
demo_losses = [0.80, 0.60, 0.45, 0.42, 0.48]
m = nn.Linear(10, 2)
print("Demo auto-checkpoint run:")
for ep, loss in enumerate(demo_losses, start=1):
    saved = auto_checkpoint(ep, m.state_dict(), loss)
    status = "SAVED" if saved else "skipped"
    print(f"  Epoch {ep} | loss={loss:.3f} | {status}")

---
## Concept 3 — Model Registry

### Why a registry?

A **model registry** is a catalog that tracks every version of a model and its current lifecycle stage: *Deprecated → Staging → Production*. Before any version touches real traffic it must be in **Staging** — where it undergoes latency and safety tests. Only after passing those gates is it promoted to **Production**.

This prevents teams from accidentally serving an untested or deprecated model.

### Pattern from the Unit 1 PDF

In [ ]:
# ── SETUP: Concept Demo ──────────────────────────────────────────────────────

model_registry = {
    "v1.0": "Deprecated - Too slow",
    "v1.1": "Production - Stable",
    "v1.2": "Staging - Undergoing latency tests"
}

def get_deployment_model(version_tag):
    status = model_registry.get(version_tag, "Version not found")
    print(f"Checking Registry: Version {version_tag} is currently {status}")
    return status

# Demo lookups
for v in ["v1.0", "v1.1", "v1.2", "v2.0"]:
    get_deployment_model(v)

---
## Concept 4 — Latency Testing with Percentile Metrics

### Why measure percentiles, not just average?

Average latency hides outliers. If 95 % of requests complete in 5 ms but 5 % take 2 seconds, users perceive the service as unreliable. The industry standard is:

- **P50** (median): typical experience
- **P95**: worst experience for 1 in 20 users — the SLA target
- **P99**: worst experience for 1 in 100 users — early warning for tail-latency issues

Most production SLAs require **P95 < 100 ms** for real-time inference.

### Pattern from the Unit 1 PDF

In [ ]:
# ── SETUP: Concept Demo ──────────────────────────────────────────────────────

def run_latency_test(model, iterations=10):
    """Run `iterations` forward passes and return (avg_ms, p95_ms)."""
    model.eval()
    latencies = []
    input_data = torch.randn(1, 10)
    for i in range(iterations):
        start_time = time.perf_counter()
        with torch.no_grad():
            _ = model(input_data)
        end_time = time.perf_counter()
        latency_ms = (end_time - start_time) * 1000
        latencies.append(latency_ms)
    avg_latency = sum(latencies) / len(latencies)
    p95_latency = sorted(latencies)[int(0.95 * len(latencies)) - 1]
    return avg_latency, p95_latency

m = nn.Linear(10, 2)
avg, p95 = run_latency_test(m, iterations=20)
print(f"Demo latency  →  avg={avg:.3f} ms  |  p95={p95:.3f} ms")

---
## Concept 5 — Safety Content Filters

### Why filter model outputs?

Language models and generative models can produce harmful content. Before any output reaches an end-user it should pass through a **content safety filter**. A simple filter checks for restricted keywords; production systems use classifiers trained on toxic content datasets.

Even a basic filter is better than nothing — it creates an auditable record of what was blocked and why.

### Pattern from the Unit 1 PDF

In [ ]:
# ── SETUP: Concept Demo ──────────────────────────────────────────────────────

def safety_test(model_output_text):
    """Return True if output is safe, False if it contains restricted content."""
    restricted_content = ["malware", "hate", "violence"]
    is_safe = True
    for word in restricted_content:
        if word in model_output_text.lower():
            is_safe = False
            break
    return is_safe

# Demo
test_outputs = [
    "The weather today is sunny and warm.",
    "Here is how to write malware for beginners.",
    "I enjoy cooking and reading books.",
    "This content promotes violence against minorities."
]
for text in test_outputs:
    result = safety_test(text)
    label = "SAFE" if result else "BLOCKED"
    print(f"  [{label}] {text[:60]}")

---
# Task 1 — Versioned Checkpoint with Simulated Training (25 points)

### Instructions

You will simulate a training loop over 5 epochs.  
For each epoch, call `auto_checkpoint` with the simulated loss below.  
After the loop, save the **best** (lowest-loss) epoch as a full versioned checkpoint at:

```
/tmp/aiat125_unit1/task1_best_checkpoint.pth
```

The saved file must contain two keys: `'model_state'` and `'metadata'`.  
The metadata dict must contain at least: `version`, `developer`, `training_loss`, `framework`.

**Simulated epoch losses** (use exactly these values):

| Epoch | Loss |
|-------|------|
| 1 | 0.72 |
| 2 | 0.55 |
| 3 | 0.38 |
| 4 | 0.41 |
| 5 | 0.36 |

> The best epoch is whichever has the lowest loss. Save that epoch's model state.

In [ ]:
# ── TASK 1 ────────────────────────────────────────────────────────────────────
# Replace every `pass` and fill in every `# TODO` below.
# Do NOT change variable names used in the assertions.

TASK1_SAVE_PATH = os.path.join(SAVE_DIR, "task1_best_checkpoint.pth")

# TODO 1a: Define a nn.Linear(10, 2) model and call it `task1_model`
task1_model = None  # YOUR CODE HERE

# Simulated epoch losses — do NOT change these
epoch_losses = [0.72, 0.55, 0.38, 0.41, 0.36]

# TODO 1b: Loop over the epoch losses.
#          Track the best (lowest) loss and which epoch it occurred at.
#          Hint: start best_loss at infinity (float('inf')) so the first
#          epoch always sets a new best.

best_loss = float('inf')   # do NOT rename this variable
best_epoch = None          # do NOT rename this variable
best_model_state = None    # do NOT rename this variable

# YOUR CODE HERE — write the training loop
pass

# TODO 1c: Build the metadata dict for the best checkpoint.
#          It must have these exact keys: version, developer, training_loss, framework
task1_metadata = None  # YOUR CODE HERE

# TODO 1d: Save the checkpoint to TASK1_SAVE_PATH using torch.save.
#          The dict must have keys 'model_state' and 'metadata'.
# YOUR CODE HERE
pass

print(f"Task 1 complete. Best epoch={best_epoch}, best_loss={best_loss}")
print(f"Checkpoint saved to: {TASK1_SAVE_PATH}")

In [ ]:
# ── ASSERTIONS: Task 1 ────────────────────────────────────────────────────────
# Run this cell after completing Task 1. It will raise AssertionError with a
# helpful message if something is wrong.

assert task1_model is not None, "task1_model must be defined (not None)"
assert isinstance(task1_model, nn.Linear), "task1_model must be nn.Linear"

assert best_epoch is not None, "best_epoch must be set inside your loop"
assert best_epoch == 5, f"Expected best_epoch=5 (loss 0.36 is the minimum), got {best_epoch}"
assert abs(best_loss - 0.36) < 1e-6, f"Expected best_loss=0.36, got {best_loss}"

assert os.path.exists(TASK1_SAVE_PATH), f"Checkpoint file not found at {TASK1_SAVE_PATH}"

ckpt = torch.load(TASK1_SAVE_PATH, weights_only=False)
assert 'model_state' in ckpt, "Checkpoint must contain key 'model_state'"
assert 'metadata' in ckpt, "Checkpoint must contain key 'metadata'"

meta = ckpt['metadata']
for key in ['version', 'developer', 'training_loss', 'framework']:
    assert key in meta, f"metadata is missing required key: '{key}'"

assert meta['training_loss'] == best_loss, (
    f"metadata['training_loss'] should equal best_loss ({best_loss}), got {meta['training_loss']}"
)

print("Task 1 PASSED — checkpoint saved with correct structure and best epoch.")

---
# Task 2 — Load and Verify a Checkpoint (25 points)

### Instructions

Implement the function `load_and_verify_checkpoint(path)` that:

1. Loads the `.pth` file at `path`.
2. Verifies that the file contains both `'model_state'` and `'metadata'` keys. If either is missing, raise a `ValueError` with a descriptive message.
3. Verifies that `metadata` contains all four required keys: `version`, `developer`, `training_loss`, `framework`. Raise `ValueError` if any are missing.
4. Returns a tuple `(model_state, metadata)`.

Then call your function on `TASK1_SAVE_PATH` and store the results in `task2_state` and `task2_meta`.

> This function is what a deployment pipeline runs when loading a model from the artifact store. The explicit verification step prevents deploying corrupted or incomplete checkpoints.

In [ ]:
# ── TASK 2 ────────────────────────────────────────────────────────────────────

REQUIRED_METADATA_KEYS = ['version', 'developer', 'training_loss', 'framework']

def load_and_verify_checkpoint(path):
    """
    Load a .pth checkpoint from `path`, verify its structure, and return
    (model_state, metadata).

    Raises:
        FileNotFoundError: if the file does not exist.
        ValueError: if required keys are missing from the checkpoint or metadata.

    Returns:
        tuple: (model_state dict, metadata dict)
    """
    # TODO 2a: Raise FileNotFoundError if path does not exist
    # YOUR CODE HERE
    pass

    # TODO 2b: Load the checkpoint with torch.load (use weights_only=False)
    checkpoint = None  # YOUR CODE HERE

    # TODO 2c: Verify 'model_state' and 'metadata' are present; raise ValueError if not
    # YOUR CODE HERE
    pass

    # TODO 2d: Verify all REQUIRED_METADATA_KEYS are in checkpoint['metadata']; raise ValueError if not
    # YOUR CODE HERE
    pass

    # TODO 2e: Return (model_state, metadata)
    # YOUR CODE HERE
    pass


# TODO 2f: Call your function on TASK1_SAVE_PATH and store results
task2_state = None  # YOUR CODE HERE
task2_meta  = None  # YOUR CODE HERE

print("Loaded metadata:", task2_meta)

In [ ]:
# ── ASSERTIONS: Task 2 ────────────────────────────────────────────────────────

assert callable(load_and_verify_checkpoint), "load_and_verify_checkpoint must be a function"

# Test that it raises FileNotFoundError on a missing file
try:
    load_and_verify_checkpoint("/tmp/this_file_does_not_exist.pth")
    assert False, "Should have raised FileNotFoundError"
except FileNotFoundError:
    pass  # expected

# Test that it raises ValueError on a corrupt checkpoint (missing keys)
corrupt_path = os.path.join(SAVE_DIR, "corrupt.pth")
torch.save({'model_state': {}}, corrupt_path)  # intentionally missing 'metadata'
try:
    load_and_verify_checkpoint(corrupt_path)
    assert False, "Should have raised ValueError for missing 'metadata' key"
except ValueError:
    pass  # expected

# Test successful load from Task 1 checkpoint
assert task2_state is not None, "task2_state must not be None"
assert task2_meta  is not None, "task2_meta must not be None"
assert isinstance(task2_state, dict), "task2_state must be a dict (model state_dict)"
assert isinstance(task2_meta,  dict), "task2_meta must be a dict"

for key in REQUIRED_METADATA_KEYS:
    assert key in task2_meta, f"task2_meta missing expected key: '{key}'"

print("Task 2 PASSED — load_and_verify_checkpoint works correctly.")

---
# Task 3 — Model Registry with Promotion Gate (25 points)

### Instructions

Build a **model registry** that enforces a deployment gate. Implement two things:

**3a.** Create `student_registry` — a dict with exactly these three entries:

| Version | Status string (must contain this word) |
|---------|----------------------------------------|
| `"v2.0"` | must contain `"Deprecated"` |
| `"v2.1"` | must contain `"Production"` |
| `"v2.2"` | must contain `"Staging"` |

**3b.** Implement `promote_to_production(registry, version)` that:
- Looks up the version in the registry.
- If the version is NOT in the registry, raises `KeyError`.
- If the version's status does NOT contain `"Staging"`, raises `ValueError` (you cannot promote a Deprecated or already-Production model).
- If the version IS in Staging, updates the registry entry to `"Production - Promoted"` and returns `True`.

**3c.** Call `promote_to_production(student_registry, "v2.2")` to demonstrate a successful promotion.

In [ ]:
# ── TASK 3 ────────────────────────────────────────────────────────────────────

# TODO 3a: Define student_registry with three version entries
student_registry = None  # YOUR CODE HERE


def promote_to_production(registry, version):
    """
    Promote a version from Staging to Production in the registry.

    Args:
        registry (dict): model registry mapping version tag → status string.
        version  (str):  version tag to promote, e.g. 'v2.2'.

    Raises:
        KeyError:   if version is not in the registry.
        ValueError: if the version's current status does not contain 'Staging'.

    Returns:
        bool: True on successful promotion.
    """
    # TODO 3b-i: Raise KeyError if version is not in registry
    # YOUR CODE HERE
    pass

    # TODO 3b-ii: Raise ValueError if current status does not contain 'Staging'
    # YOUR CODE HERE
    pass

    # TODO 3b-iii: Update registry entry to 'Production - Promoted' and return True
    # YOUR CODE HERE
    pass


# TODO 3c: Promote v2.2 and print the updated registry
# YOUR CODE HERE
pass

print("Updated registry:", student_registry)

In [ ]:
# ── ASSERTIONS: Task 3 ────────────────────────────────────────────────────────

assert isinstance(student_registry, dict), "student_registry must be a dict"
assert len(student_registry) == 3, f"student_registry must have 3 entries, got {len(student_registry)}"

for v in ["v2.0", "v2.1", "v2.2"]:
    assert v in student_registry, f"student_registry must contain version '{v}'"

assert "Deprecated" in student_registry["v2.0"], "v2.0 status must contain 'Deprecated'"
assert "Production" in student_registry["v2.1"], "v2.1 status must contain 'Production'"

# After promotion, v2.2 should be Production
assert "Production" in student_registry["v2.2"], (
    f"After promotion, v2.2 status should contain 'Production', got: {student_registry['v2.2']}"
)

# Test KeyError for unknown version
try:
    promote_to_production(student_registry, "v9.9")
    assert False, "Should have raised KeyError for unknown version"
except KeyError:
    pass  # expected

# Test ValueError for non-Staging version (v2.1 is already Production)
try:
    promote_to_production(student_registry, "v2.1")
    assert False, "Should have raised ValueError — v2.1 is already Production, not Staging"
except ValueError:
    pass  # expected

# Test ValueError for Deprecated version
try:
    promote_to_production(student_registry, "v2.0")
    assert False, "Should have raised ValueError — v2.0 is Deprecated, not Staging"
except ValueError:
    pass  # expected

print("Task 3 PASSED — registry and promotion gate work correctly.")

---
# Task 4 — Latency Benchmark with P50 / P95 / P99 (25 points)

### Instructions

Run a **100-iteration** latency benchmark on a `nn.Linear(10, 2)` model and compute three percentile metrics.

**4a.** Create `benchmark_model = nn.Linear(10, 2)` and set it to eval mode.

**4b.** Run exactly **100** forward passes using `time.perf_counter()` to measure each one. Store all latencies (in milliseconds) in a list called `latency_results`.

**4c.** Compute:
- `p50` — the 50th percentile (median): index `int(0.50 * 100) - 1` of the sorted list
- `p95` — the 95th percentile: index `int(0.95 * 100) - 1` of the sorted list
- `p99` — the 99th percentile: index `int(0.99 * 100) - 1` of the sorted list

**4d.** Assert that `p95 < 100` ms (the production SLA). If this fails on your machine, add a short warm-up loop of 5 iterations before the timed loop.

> **Why P95 < 100 ms?** User research shows that response times above 100 ms are perceptible. Above 1000 ms users abandon the request. For interactive AI features the industry targets ≤ 100 ms at P95.

In [ ]:
# ── TASK 4 ────────────────────────────────────────────────────────────────────

NUM_ITERATIONS = 100

# TODO 4a: Define benchmark_model as nn.Linear(10, 2) and set to eval mode
benchmark_model = None  # YOUR CODE HERE

# TODO 4b: (Optional) warm-up — run 5 passes WITHOUT recording latency
#          This removes JIT / cache cold-start noise from the benchmark.
# YOUR CODE HERE
pass

# TODO 4c: Timed loop — run NUM_ITERATIONS forward passes.
#          Measure each with time.perf_counter(), convert to ms, append to latency_results.
latency_results = []  # do NOT rename this variable
input_tensor = torch.randn(1, 10)  # fixed input for all iterations

# YOUR CODE HERE
pass

# TODO 4d: Sort latency_results and compute p50, p95, p99
sorted_latencies = None  # YOUR CODE HERE
p50 = None  # YOUR CODE HERE
p95 = None  # YOUR CODE HERE
p99 = None  # YOUR CODE HERE

print(f"Latency Benchmark ({NUM_ITERATIONS} iterations)")
print(f"  P50 (median) : {p50:.4f} ms")
print(f"  P95          : {p95:.4f} ms")
print(f"  P99          : {p99:.4f} ms")
print(f"  SLA (P95 < 100 ms): {'PASS' if p95 is not None and p95 < 100 else 'FAIL'}")

In [ ]:
# ── ASSERTIONS: Task 4 ────────────────────────────────────────────────────────

assert benchmark_model is not None, "benchmark_model must be defined"
assert isinstance(benchmark_model, nn.Linear), "benchmark_model must be nn.Linear"
assert not benchmark_model.training, "benchmark_model must be in eval mode (call model.eval())"

assert isinstance(latency_results, list), "latency_results must be a list"
assert len(latency_results) == NUM_ITERATIONS, (
    f"latency_results must have {NUM_ITERATIONS} entries, got {len(latency_results)}"
)
assert all(isinstance(x, float) for x in latency_results), "All latency values must be floats (ms)"
assert all(x > 0 for x in latency_results), "All latency values must be positive"

assert p50 is not None, "p50 must be computed"
assert p95 is not None, "p95 must be computed"
assert p99 is not None, "p99 must be computed"

assert p50 <= p95 <= p99, f"Percentile ordering must hold: p50 ≤ p95 ≤ p99, got {p50:.4f} / {p95:.4f} / {p99:.4f}"

assert p95 < 100, (
    f"p95 latency is {p95:.4f} ms — exceeds the 100 ms SLA. "
    "Add a 5-iteration warm-up loop before the timed loop to fix cold-start noise."
)

print("Task 4 PASSED — latency benchmark complete, P95 < 100 ms SLA met.")

---
## Bonus — Safety Filter Integration

The `safety_test` function from Concept 5 is already defined above. This section shows how it would be wired into a deployment pipeline.

Run this cell to see the safety filter in action — no code changes needed.

In [ ]:
# ── BONUS: Safety Filter Pipeline Demo ───────────────────────────────────────
# This simulates what happens AFTER a model produces text output:
# the output is checked by safety_test before being returned to the user.

def safe_inference_pipeline(model_output_text):
    """Simulated pipeline that runs safety filter before returning output."""
    if safety_test(model_output_text):
        return {"status": "OK", "output": model_output_text}
    else:
        return {"status": "BLOCKED", "output": "[Content removed by safety filter]"}

sample_outputs = [
    "Saudi Arabia produced 10 million barrels of oil in 2023.",
    "Here is a tutorial on creating malware for educational purposes.",
    "The capital of France is Paris and it has many museums.",
    "This article promotes hate against a specific religion."
]

print("Safety Filter Pipeline Results:")
print("-" * 60)
for text in sample_outputs:
    result = safe_inference_pipeline(text)
    print(f"[{result['status']:7s}] {result['output'][:55]}")

---
# Final Deployment Gate — Summary Report

Run this cell last. It will check all four tasks and print a PASS / FAIL report.

All four tasks must PASS before submitting.

In [ ]:
# ── DEPLOYMENT GATE — Final Summary ──────────────────────────────────────────

results = {}

# ── Task 1 ──
try:
    assert task1_model is not None and isinstance(task1_model, nn.Linear)
    assert best_epoch == 5 and abs(best_loss - 0.36) < 1e-6
    ckpt1 = torch.load(TASK1_SAVE_PATH, weights_only=False)
    assert 'model_state' in ckpt1 and 'metadata' in ckpt1
    for k in ['version', 'developer', 'training_loss', 'framework']:
        assert k in ckpt1['metadata']
    results['Task 1 - Versioned Checkpoint (25 pts)'] = 'PASS'
except Exception as e:
    results['Task 1 - Versioned Checkpoint (25 pts)'] = f'FAIL — {e}'

# ── Task 2 ──
try:
    assert callable(load_and_verify_checkpoint)
    assert task2_state is not None and isinstance(task2_state, dict)
    assert task2_meta  is not None and isinstance(task2_meta,  dict)
    for k in ['version', 'developer', 'training_loss', 'framework']:
        assert k in task2_meta
    results['Task 2 - Load & Verify Checkpoint (25 pts)'] = 'PASS'
except Exception as e:
    results['Task 2 - Load & Verify Checkpoint (25 pts)'] = f'FAIL — {e}'

# ── Task 3 ──
try:
    assert isinstance(student_registry, dict) and len(student_registry) == 3
    assert 'Deprecated' in student_registry.get('v2.0', '')
    assert 'Production' in student_registry.get('v2.1', '')
    assert 'Production' in student_registry.get('v2.2', '')
    results['Task 3 - Model Registry & Promotion Gate (25 pts)'] = 'PASS'
except Exception as e:
    results['Task 3 - Model Registry & Promotion Gate (25 pts)'] = f'FAIL — {e}'

# ── Task 4 ──
try:
    assert benchmark_model is not None and isinstance(benchmark_model, nn.Linear)
    assert not benchmark_model.training
    assert len(latency_results) == 100
    assert p50 is not None and p95 is not None and p99 is not None
    assert p50 <= p95 <= p99
    assert p95 < 100
    results['Task 4 - Latency Benchmark P50/P95/P99 (25 pts)'] = 'PASS'
except Exception as e:
    results['Task 4 - Latency Benchmark P50/P95/P99 (25 pts)'] = f'FAIL — {e}'

# ── Print Report ──
print("=" * 65)
print("  AIAT 125 — Unit 1 Guided Lab: DEPLOYMENT GATE REPORT")
print("=" * 65)
total_pass = 0
for task, status in results.items():
    icon = "✓" if status == 'PASS' else "✗"
    print(f"  {icon}  {task}: {status}")
    if status == 'PASS':
        total_pass += 1
print("=" * 65)
score = total_pass * 25
print(f"  Score: {score} / 100  ({total_pass}/4 tasks passed)")
print("=" * 65)
if total_pass == 4:
    print("  ALL GATES PASSED — This model is cleared for deployment.")
else:
    print("  GATES NOT FULLY PASSED — Fix failing tasks above and re-run.")
print("=" * 65)

---
## Closing Takeaway

Before moving to the next notebook, answer these questions in your own words (no code needed):

1. **Versioning**: What information would you add to the metadata dict to make rollback decisions easier in a team setting?

2. **Checkpointing**: Why is `best_loss = float('inf')` a better starting value than `best_loss = 0.50` for a general-purpose training loop?

3. **Registry**: What additional lifecycle stages might a large organisation add between Staging and Production (e.g. shadow traffic, canary)?

4. **Latency**: If your P99 latency is 800 ms but P95 is 40 ms, what does that tell you about your system?

5. **Safety**: Why is a keyword blocklist not sufficient as a production safety filter, and what would you add?

---
**Next**: Proceed to the next notebook in the unit README for API serving fundamentals.

**References:**
- [PyTorch Saving and Loading Models](https://pytorch.org/tutorials/beginner/saving_loading_models.html)
- [MLflow Model Registry Concepts](https://mlflow.org/docs/latest/model-registry.html)
- [Google SRE Book — Latency and Percentiles](https://sre.google/sre-book/monitoring-distributed-systems/)
- [Responsible AI Practices — Google](https://ai.google/responsibilities/responsible-ai-practices/)